# Walmart Store Sales Analysis — Dokumentacja projektu

**Cel projektu**

Celem analizy jest zbadanie wzorców sprzedaży w sieci sklepów Walmart w latach 2010–2012 i odpowiedź na pytanie: jakie czynniki (sezonowość, święta, promocje, typ sklepu, warunki makroekonomiczne) wpływają na tygodniową sprzedaż, i jak ta wiedza może wspierać lepsze planowanie zapasów i prognozowanie popytu?
Źródło danych

Dane pochodzą z konkursu Kaggle "Walmart Recruiting — Store Sales Forecasting" (2014). Zbiór zawiera rzeczywiste dane sprzedażowe Walmart i składa się z trzech plików: train.csv (421 570 wierszy tygodniowej sprzedaży per sklep i dział), features.csv (dane zewnętrzne: temperatura, cena paliwa, markdowny promocyjne, CPI, bezrobocie) oraz stores.csv (typ i wielkość każdego z 45 sklepów). Dane obejmują 45 sklepów, 81 działów i 143 tygodnie (luty 2010 – październik 2012).

**Pytania badawcze**

1. Czy i jak silna jest sezonowość świąteczna w sprzedaży Walmart, i czy jest ona jednolita między typami sklepów?
2. Które sklepy i działy generują największą część całkowitej sprzedaży (analiza ABC/Pareto)?
3. Czy i w jakim stopniu poziom promocji (MarkDown) wpływa na sprzedaż tygodniową?
4. Czy warunki makroekonomiczne (bezrobocie, CPI, cena paliwa) korelują ze sprzedażą, i czy ta korelacja jest przyczynowa?

**Narzędzia i metodologia**

Projekt realizowany jest w trzech etapach z użyciem różnych narzędzi: SQL (eksploracja danych, czyszczenie, agregacje, łączenie tabel) → Excel (tabele przestawne, analiza ABC/Pareto, weryfikacja hipotez, prognoza liniowa) → Power BI (interaktywny dashboard z czterema stronami tematycznymi). Każdy etap buduje na wynikach poprzedniego, a decyzje analityczne są dokumentowane wraz z uzasadnieniem.
Struktura dokumentacji

Dokumentacja jest podzielona na trzy sekcje odpowiadające fazom projektu: WALMART (SQL w DataLab), EXCEL i POWER BI. Każdy krok zawiera zapytanie lub działanie, wynik oraz wniosek analityczny lub decyzję dotyczącą dalszego postępowania z danymi.

# SQL

In [16]:
SELECT 
	MIN(date) AS min_date, 
	MAX(date) AS max_date, 
	COUNT(DISTINCT Store) AS stores_qty, 
	COUNT(DISTINCT Dept) AS depts_qty
FROM train.csv;

,min_date,max_date,stores_qty,depts_qty
0,2010-02-05 00:00:00+00:00,2012-10-26 00:00:00+00:00,45,81


Krok 1: Zakres danych i wymiary zbioru

Tabela train zawiera dane sprzedażowe z 143 tygodni, od 2010-02-05 do 2012-10-26. Obejmuje 45 unikalnych sklepów oraz 81 unikalnych działów (Dept). Teoretyczna maksymalna liczba kombinacji sklep–dział–tydzień to ok. 521 tys. wierszy, natomiast tabela zawiera 421 570 wierszy, co wskazuje, że nie każdy dział działał w każdym sklepie przez cały okres (typowe dla danych retail — działy bywają otwierane, zamykane lub nieaktywne w danych okresach).

In [15]:
SELECT 
	MIN(weekly_sales) AS min_weekly_sales, 
	MAX(weekly_sales) AS max_weekly_sales, 
	AVG(weekly_sales) AS avg_weekly_sales, 
	SUM(CASE WHEN weekly_sales < 0 THEN 1 ELSE 0 END) AS negative_values_qty
FROM train.csv;

,min_weekly_sales,max_weekly_sales,avg_weekly_sales,negative_values_qty
0,-4988.94,693099.36,15981.258123,1285.0


Krok 2: Sprawdzenie jakości danych: Weekly_Sales

Zakres wartości: od -4 988,94 do 693 099,36, średnia ≈ 15 981,26. Znaleziono 1 285 wierszy z ujemną wartością sprzedaży tygodniowej, co stanowi ok. 0,3% danych. Ujemne wartości najpewniej oznaczają zwroty towaru przewyższające sprzedaż w danym tygodniu (a nie błąd w danych) i wymagają decyzji: czy wykluczyć je z analizy trendów sprzedażowych, czy zostawić jako realny sygnał biznesowy (np. dział z wysoką stopą zwrotów).

In [18]:
SELECT 
	dept, 
	COUNT(*) AS negative_values_qty
FROM train.csv
WHERE weekly_sales < 0
GROUP BY dept
ORDER BY negative_values_qty DESC;

,Dept,negative_values_qty
0,47,254
1,18,180
2,54,146
3,19,87
4,94,77
5,80,68
6,49,67
7,59,44
8,72,34
9,78,33


Krok 3: Koncentracja ujemnych wartości Weekly_Sales po działach

Ujemne wartości sprzedaży tygodniowej nie są rozproszone losowo — silnie koncentrują się w kilku działach. Dept 47 odpowiada za 254 z 1 285 przypadków (~20%), a top 3 działy (47, 18, 54) łącznie za ponad 45% wszystkich ujemnych wierszy. To sugeruje, że ujemne wartości to prawdopodobnie realny wzorzec biznesowy (np. działy z wysoką stopą zwrotów, sezonowość, lub specyfika kategorii produktowej), a nie losowy błąd w danych. Decyzja: zachowujemy ujemne wartości w analizie, ale oznaczamy je jako odrębną kategorię do zbadania, zamiast usuwać jako "błędne dane".

In [19]:
SELECT 
    strftime('%Y-%m', date) AS year_month,
    SUM(weekly_sales) AS total_sales
FROM train.csv
GROUP BY year_month
ORDER BY year_month ASC;

,year_month,total_sales
0,2010-02,1.903330e+08
1,2010-03,1.819198e+08
2,2010-04,2.314124e+08
3,2010-05,1.867109e+08
4,2010-06,1.922462e+08
5,2010-07,2.325801e+08
6,2010-08,1.876401e+08
7,2010-09,1.772679e+08
8,2010-10,2.171618e+08
9,2010-11,2.028534e+08


Krok 4: Trend sprzedaży miesięcznej

Sprzedaż zagregowana miesięcznie pokazuje wyraźną sezonowość. Grudzień konsekwentnie jest szczytem sprzedaży (np. grudzień 2010: ~288,8 mln, najwyższy wynik w pierwszym roku danych), a styczeń jest najsłabszym miesiącem (styczeń 2011: ~163,7 mln, spadek o ~43% względem grudnia). Wzorzec ten potwierdza silny wpływ okresu świątecznego na sprzedaż i będzie kluczowym elementem do uwzględnienia w prognozowaniu popytu oraz planowaniu zapasów.

In [21]:
SELECT 
	type,
	AVG(weekly_sales) AS avg_weekly_sales
FROM train.csv t
INNER JOIN stores.csv s ON t.store = s.store
GROUP BY type;

,Type,avg_weekly_sales
0,A,20099.568043
1,B,12237.075977
2,C,9519.532538


Krok 5: Średnia sprzedaż tygodniowa według typu sklepu

Po połączeniu tabel train i stores (JOIN po kolumnie Store), średnia sprzedaż tygodniowa rośnie wraz z typem/wielkością sklepu: Typ A ~20 100, Typ B ~12 237, Typ C ~9 519. Różnica między typem A i C to ponad 2-krotność, co potwierdza silną korelację między powierzchnią sklepu a wolumenem sprzedaży. Ten wynik stanowi punkt odniesienia (baseline) do dalszej analizy — np. sprawdzenia, czy sezonowość świąteczna wpływa proporcjonalnie czy nieproporcjonalnie na różne typy sklepów.

In [25]:
SELECT 
	type,
	isholiday,
	AVG(weekly_sales) AS avg_weekly_sales
FROM train.csv t
INNER JOIN stores.csv s ON t.store = s.store
GROUP BY type, isholiday
ORDER BY avg_weekly_sales DESC;

,Type,IsHoliday,avg_weekly_sales
0,A,True,21297.517824
1,A,False,20008.746759
2,B,True,13346.164062
3,B,False,12153.067752
4,C,True,9532.963131
5,C,False,9518.528116


Krok 6: Wpływ świąt na sprzedaż w zależności od typu sklepu

Połączenie train ze stores i grupowanie po Type + IsHoliday pokazuje, że efekt świąteczny nie jest jednolity między typami sklepów. Sklepy typu B reagują na święta najsilniej (wzrost średniej sprzedaży o ~9,8%), typ A umiarkowanie (~6,4%), natomiast typ C praktycznie nie wykazuje wzrostu sprzedaży w okresie świątecznym (~0,1%). To sugeruje, że duże/średnie sklepy korzystają na ruchu świątecznym znacznie bardziej niż najmniejsze placówki — możliwe przyczyny to inny asortyment, mniejsza ekspozycja na zakupy prezentowe, lub odmienna baza klientów. Wniosek ten jest istotny dla planowania zapasów: zwiększanie poziomu zapasów przed świętami powinno być zróżnicowane według typu sklepu, a nie jednolite.

In [31]:
SELECT 
	MIN(date) AS first_markdown1_date
FROM features.csv
WHERE markdown1 IS NOT NULL;

,first_markdown1_date
0,2010-02-05 00:00:00+00:00


In [30]:
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN markdown1 IS NOT NULL THEN 1 ELSE 0 END) AS markdown1_filled,
    SUM(CASE WHEN markdown1 IS NOT NULL AND date < '2011-11-01' THEN 1 ELSE 0 END) AS markdown1_filled_before_nov2011
FROM features.csv;

,total_rows,markdown1_filled,markdown1_filled_before_nov2011
0,8190,8190.0,4095.0


In [29]:
SELECT DISTINCT markdown1
FROM features.csv
LIMIT 10;

,MarkDown1
0,NA
1,4640.65
2,2725.36
3,5762.1
4,5183.29
5,4139.87
6,1164.46
7,9873.33
8,10331.04
9,4298.16


Krok 7: Wykryty problem z typami danych w kolumnach MarkDown

Początkowe zapytanie filtrujące MarkDown1 IS NOT NULL zwróciło 8190 z 8190 wierszy (100%), co było niespójne z wcześniejszą obserwacją dużej liczby NA w podglądzie danych. Weryfikacja przez SELECT DISTINCT markdown1 ujawniła, że kolumna została wczytana jako typ tekstowy (string), a brakujące wartości to literalny string "NA", nie prawdziwy SQL NULL. To oznacza, że standardowe filtrowanie IS NOT NULL nie wykrywa tych braków. Wymagana poprawka: filtrowanie po markdown1 != 'NA' oraz konwersja kolumny na typ numeryczny (CAST) przed dalszymi obliczeniami matematycznymi.

In [32]:
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN markdown1 != 'NA' THEN 1 ELSE 0 END) AS markdown1_filled
FROM features.csv;

,total_rows,markdown1_filled
0,8190,4032.0


Krok 8: Korekta filtrowania i prawdziwy wskaźnik wypełnienia MarkDown1

Po poprawce filtra (markdown1 != 'NA' zamiast IS NOT NULL) okazuje się, że 4 032 z 8 190 wierszy (~49%) ma realną wartość markdown1. To potwierdza wcześniejsze założenie, że dane o promocjach (markdownach) zaczęto zbierać dopiero w połowie okresu objętego datasetem, a nie od samego początku — co jest typowe dla tego konkursu Kaggle (markdowny wprowadzono w listopadzie 2011). Decyzja: analiza wpływu markdownów na sprzedaż będzie ograniczona tylko do okresu, w którym dane są dostępne, aby nie zniekształcić wyników brakującymi danymi z wcześniejszego okresu.

In [33]:
SELECT MIN(date) AS first_markdown_date
FROM features.csv
WHERE markdown1 != 'NA';

,first_markdown_date
0,2011-11-11 00:00:00+00:00


Krok 9: Ustalenie rzeczywistego okresu dostępności danych MarkDown

Po poprawce filtra, najwcześniejsza data z realną wartością markdown1 to 2011-11-11. Potwierdza to, że dane o promocjach (MarkDown1-5) obejmują tylko ostatni rok zbioru (listopad 2011 – październik 2012), a nie cały okres 2010-2012. Każda analiza wpływu markdownów na sprzedaż musi być ograniczona do tego podzbioru czasowego, w przeciwnym razie wyniki będą obciążone brakiem danych z wcześniejszego okresu (selection bias).

In [34]:
SELECT 
    CASE WHEN CAST(f.markdown1 AS REAL) > 0 THEN 'Active promotion' ELSE 'No promotion' END AS markdown_status,
    AVG(t.weekly_sales) AS avg_weekly_sales
FROM train.csv t
JOIN features.csv f ON t.store = f.store AND t.date = f.date
WHERE f.markdown1 != 'NA' AND f.date >= '2011-11-11'
GROUP BY markdown_status;

,markdown_status,avg_weekly_sales
0,Active promotion,16215.489727


In [36]:
SELECT 
    MIN(CAST(markdown1 AS REAL)) AS min_markdown1,
    MAX(CAST(markdown1 AS REAL)) AS max_markdown1,
    COUNT(*) AS rows_checked
FROM features.csv
WHERE markdown1 != 'NA';

,min_markdown1,max_markdown1,rows_checked
0,-2781.449951,103184.976562,4032


Krok 10: Wykryta anomalia — ujemne wartości MarkDown1

Zakres wartości markdown1 (po konwersji na liczbę) to od -2 781,45 do 103 184,98, na 4 032 wierszach z realną wartością. Obecność wartości ujemnych jest nietypowa dla kolumny reprezentującej wielkość zniżki promocyjnej i wymaga dalszego sprawdzenia — może oznaczać korektę/błąd w danych, lub specyficzny sposób raportowania przez Walmart (np. korekta nadpłaconej promocji z poprzedniego okresu). Decyzja do podjęcia: sprawdzić liczbę takich przypadków i rozważyć ich wykluczenie lub osobne oznaczenie w dalszej analizie.

In [37]:
SELECT 
    SUM(CASE WHEN CAST(markdown1 AS REAL) < 0 THEN 1 ELSE 0 END) AS negative_count,
    SUM(CASE WHEN CAST(markdown1 AS REAL) = 0 THEN 1 ELSE 0 END) AS zero_count,
    SUM(CASE WHEN CAST(markdown1 AS REAL) > 0 THEN 1 ELSE 0 END) AS positive_count
FROM features.csv
WHERE markdown1 != 'NA';

,negative_count,zero_count,positive_count
0,4.0,0.0,4028.0


Krok 11: Skala anomalii ujemnych wartości MarkDown1

Spośród 4032 wierszy z realną wartością markdown1, tylko 4 wiersze (0,1%) mają wartość ujemną, a 0 wierszy ma wartość równą zero — czyli każda promocja w danych ma jakąś rzeczywistą skalę zniżki. Ujemne wartości są zbyt rzadkie, by analizować je jako odrębny wzorzec biznesowy; najpewniej to błędy w danych źródłowych (np. korekta błędnie wprowadzonej wcześniejszej promocji). Decyzja: te 4 wiersze zostaną wykluczone z analizy wpływu markdownów na sprzedaż jako nieistotny statystycznie szum, a wniosek końcowy brzmi: w analizowanym okresie każda zarejestrowana promocja (MarkDown1) miała rzeczywisty, dodatni wpływ na cenę/zniżkę — nie ma okresu "zerowej promocji" w danych z aktywnym raportowaniem.

In [39]:
SELECT 
    CASE 
        WHEN CAST(f.markdown1 AS REAL) < 5000 THEN 'Low markdown (<5000)'
        WHEN CAST(f.markdown1 AS REAL) < 15000 THEN 'Medium markdown (5000-15000)'
        ELSE 'High markdown (>15000)'
    END AS markdown_tier,
    COUNT(*) AS row_qty,
    AVG(t.weekly_sales) AS avg_weekly_sales
FROM train.csv t
JOIN features.csv f ON t.store = f.store AND t.date = f.date
WHERE f.markdown1 != 'NA' AND CAST(f.markdown1 AS REAL) >= 0
GROUP BY markdown_tier
ORDER BY avg_weekly_sales DESC;

,markdown_tier,row_qty,avg_weekly_sales
0,High markdown (>15000),15613,19318.063566
1,Mid markdown (5000-15000),63917,18450.024995
2,Low markdown (<5000),71151,13527.329673


Krok 12: Wpływ wysokości promocji (MarkDown1) na sprzedaż

Wynik pokazuje rosnący trend: Low (<5000) → śr. sprzedaż 13 527 | Mid (5000–15000) → 18 450 (+36%) | High (>15000) → 19 319 (+4,7% względem Mid). Wzrost sprzedaży spłaszcza się przy najwyższych wartościach promocji, co sugeruje malejącą efektywność marginalną dużych zniżek. Ograniczenie: to korelacja, nie dowód związku przyczynowo-skutkowego — wysokie markdowny mogą się nakładać z naturalnie silniejszymi okresami sprzedażowymi (np. przedświąteczne), co wymagałoby dalszej analizy (np. kontrola za pomocą IsHoliday) do pełnego rozdzielenia tych efektów.

In [ ]:
SELECT 
    MIN(cpi) AS min_cpi,
    MAX(cpi) AS max_cpi,
    AVG(cpi) AS avg_cpi,
    MIN(unemployment) AS min_unemployment,
    MAX(unemployment) AS max_unemployment,
    AVG(unemployment) AS avg_unemployment
FROM features.csv;

Error: Binder Error: No function matches the given name and argument types 'avg(VARCHAR)'. You might need to add explicit type casts.
	Candidate functions:
	avg(DECIMAL) -> DECIMAL
	avg(SMALLINT) -> DOUBLE
	avg(INTEGER) -> DOUBLE
	avg(BIGINT) -> DOUBLE
	avg(HUGEINT) -> DOUBLE
	avg(INTERVAL) -> INTERVAL
	avg(DOUBLE) -> DOUBLE
	avg(TIMESTAMP) -> TIMESTAMP
	avg(TIMESTAMP WITH TIME ZONE) -> TIMESTAMP WITH TIME ZONE
	avg(TIME) -> TIME
	avg(TIME WITH TIME ZONE) -> TIME WITH TIME ZONE


LINE 4:     AVG(cpi) AS avg_cpi,
            ^

In [ ]:
SELECT DISTINCT cpi
FROM features.csv
WHERE CAST(cpi AS REAL) IS NULL
LIMIT 5;

Error: Conversion Error: Could not convert string 'NA' to FLOAT when casting from source column CPI

LINE 3: WHERE CAST(cpi AS REAL) IS NULL
              ^

In [40]:
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN cpi = 'NA' THEN 1 ELSE 0 END) AS cpi_na_count,
    SUM(CASE WHEN unemployment = 'NA' THEN 1 ELSE 0 END) AS unemployment_na_count
FROM features.csv;

,total_rows,cpi_na_count,unemployment_na_count
0,8190,585.0,585.0


Krok 13: Problem typu danych w kolumnach CPI i Unemployment

Próba policzenia AVG/MIN/MAX na kolumnach cpi i unemployment zwróciła błąd (No function matches AVG(VARCHAR)), co ujawniło, że obie kolumny zostały wczytane jako tekst, podobnie jak wcześniej MarkDown1. Sprawdzenie wykazało, że 585 z 8190 wierszy (~7,1%) ma wartość 'NA' w obu kolumnach jednocześnie — identyczna liczba w obu kolumnach sugeruje, że są to te same wiersze (np. dane dla konkretnego, późniejszego okresu czasu lub konkretnych sklepów, dla których wskaźniki makroekonomiczne nie zostały jeszcze zaktualizowane w źródle). Wymagana poprawka: filtrowanie != 'NA' przed użyciem CAST, ponieważ DuckDB (w przeciwieństwie do niektórych innych silników SQL) rzuca błąd przy próbie konwersji nieprawidłowego tekstu, zamiast cicho zwracać NULL.

In [41]:
SELECT 
    MIN(date) AS min_date_na,
    MAX(date) AS max_date_na
FROM features.csv
WHERE cpi = 'NA';

,min_date_na,max_date_na
0,2013-05-03 00:00:00+00:00,2013-07-26 00:00:00+00:00


Krok 14: Źródło brakujących wartości CPI/Unemployment — dane wykraczające poza okres treningowy

Brakujące wartości CPI i Unemployment (585 wierszy) odpowiadają dokładnie okresowi 2013-05-03 do 2013-07-26, który wykracza poza zakres tabeli train (kończącej się 2012-10-26). Oznacza to, że features.csv zawiera dodatkowe dane z okresu testowego konkursu Kaggle, dla którego wskaźniki makroekonomiczne nie były jeszcze opublikowane w momencie zbierania danych. Wniosek: przy złączeniu (INNER JOIN) tabeli features z train, te wiersze automatycznie nie wejdą do wyniku, ponieważ train nie posiada odpowiadających dat. Problem z brakującymi CPI/Unemployment jest więc nieistotny dla głównej analizy korelacji sprzedaż–ekonomia, o ile używamy INNER JOIN (a nie LEFT JOIN od strony features).

In [43]:
SELECT COUNT(*) AS na_rows_after_join
FROM train.csv t
INNER JOIN features.csv f ON t.store = f.store AND t.date = f.date
WHERE f.cpi = 'NA';

,na_rows_after_join
0,0


Krok 15: Weryfikacja — brak wpływu problemu CPI/Unemployment na analizę główną

Po złączeniu train z features przez INNER JOIN, zapytanie filtrujące cpi = 'NA' zwróciło 0 wierszy, co potwierdza wcześniejszą hipotezę: wszystkie wiersze z brakującym CPI/Unemployment pochodzą z okresu poza zakresem danych treningowych i są naturalnie wyeliminowane przez logikę złączenia. Można bezpiecznie przejść do konwersji typu i liczenia korelacji bez dodatkowego filtrowania 'NA' — wystarczy CAST po złączeniu z train.

In [44]:
SELECT 
    MIN(CAST(f.cpi AS REAL)) AS min_cpi,
    MAX(CAST(f.cpi AS REAL)) AS max_cpi,
    AVG(CAST(f.cpi AS REAL)) AS avg_cpi,
    MIN(CAST(f.unemployment AS REAL)) AS min_unemployment,
    MAX(CAST(f.unemployment AS REAL)) AS max_unemployment,
    AVG(CAST(f.unemployment AS REAL)) AS avg_unemployment
FROM train.csv t
JOIN features.csv f ON t.store = f.store AND t.date = f.date;

,min_cpi,max_cpi,avg_cpi,min_unemployment,max_unemployment,avg_unemployment
0,126.064003,227.232803,171.201947,3.879,14.313,7.960289


Krok 16: Statystyki CPI i Unemployment w analizowanym okresie

Po połączeniu z train i konwersji typów, CPI w danych mieści się w zakresie 126,1–227,2 (średnia ~171,2), a bezrobocie w zakresie 3,9%–14,3% (średnia ~7,6%). Szeroki rozstrzał wynika z tego, że wskaźniki są przypisane do regionów poszczególnych sklepów, nie jednego krajowego wskaźnika — różne lokalizacje miały różną sytuację gospodarczą w tym okresie (część regionów silniej dotknięta skutkami kryzysu z 2008 roku). Wartości są spójne z realnymi danymi makroekonomicznymi USA z lat 2010–2012.

In [45]:
SELECT 
    CASE 
        WHEN CAST(f.unemployment AS REAL) < 7 THEN 'Low unemployment (<7%)'
        WHEN CAST(f.unemployment AS REAL) < 9 THEN 'Medium unemployment (7-9%)'
        ELSE 'High unemployment (>9%)'
    END AS unemployment_tier,
    COUNT(*) AS row_qty,
    AVG(t.weekly_sales) AS avg_weekly_sales
FROM train.csv t
JOIN features.csv f ON t.store = f.store AND t.date = f.date
GROUP BY unemployment_tier
ORDER BY avg_weekly_sales DESC;

,unemployment_tier,row_qty,avg_weekly_sales
0,Medium unemployment (7-9%),232452,17115.718053
1,Low unemployment (<7%),118623,15103.714344
2,High unemployment (>9%),70495,13717.117352


Krok 17: Wpływ bezrobocia na sprzedaż — wynik nieliniowy i niejednoznaczny

Podział na przedziały bezrobocia pokazuje brak prostej, liniowej zależności między stopą bezrobocia a sprzedażą: grupa ze średnim bezrobociem (7-9%) ma najwyższą średnią sprzedaż (17 116), wyższą niż grupa z niskim bezrobociem (<7%, śr. 15 104). To prawdopodobnie efekt pomieszania zmiennych (confounding) — różne poziomy bezrobocia mogą odpowiadać różnym regionom geograficznym, w których rozkład typów/wielkości sklepów jest nierównomierny, a nie czystemu wpływowi sytuacji ekonomicznej na zachowania zakupowe. Wniosek: surowa korelacja bezrobocie-sprzedaż jest niewystarczająca do wnioskowania przyczynowego; wymagana byłaby analiza kontrolująca typ/wielkość sklepu (np. w Excelu przez tabelę przestawną z dwoma wymiarami) dla rzetelnej interpretacji.

In [46]:
SELECT 
    t.store,
    t.dept,
    t.date,
    t.weekly_sales,
    t.isholiday,
    s.type AS store_type,
    s.size AS store_size,
    f.temperature,
    f.fuel_price,
    f.markdown1,
    f.markdown2,
    f.markdown3,
    f.markdown4,
    f.markdown5,
    CAST(f.cpi AS REAL) AS cpi,
    CAST(f.unemployment AS REAL) AS unemployment
FROM train.csv t
INNER JOIN features.csv f 
    ON t.store = f.store AND t.date = f.date
INNER JOIN stores.csv s 
    ON t.store = s.store
ORDER BY t.store, t.dept, t.date;

,Store,Dept,Date,Weekly_Sales,IsHoliday,store_type,store_size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,cpi,unemployment
0,1,1,2010-02-05 00:00:00+00:00,24924.50,False,A,151315,42.31,2.572,NA,NA,NA,NA,NA,211.096359,8.106
1,1,1,2010-02-12 00:00:00+00:00,46039.49,True,A,151315,38.51,2.548,NA,NA,NA,NA,NA,211.242172,8.106
2,1,1,2010-02-19 00:00:00+00:00,41595.55,False,A,151315,39.93,2.514,NA,NA,NA,NA,NA,211.289139,8.106
3,1,1,2010-02-26 00:00:00+00:00,19403.54,False,A,151315,46.63,2.561,NA,NA,NA,NA,NA,211.319641,8.106
4,1,1,2010-03-05 00:00:00+00:00,21827.90,False,A,151315,46.50,2.625,NA,NA,NA,NA,NA,211.350143,8.106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28 00:00:00+00:00,508.37,False,B,118221,64.88,3.997,4556.61,20.64,1.5,1601.01,3288.25,192.013565,8.684
421566,45,98,2012-10-05 00:00:00+00:00,628.10,False,B,118221,64.89,3.985,5046.74,NA,18.82,2253.43,2340.01,192.170410,8.667
421567,45,98,2012-10-12 00:00:00+00:00,1061.02,False,B,118221,54.47,4.000,1956.28,NA,7.89,599.32,3990.54,192.327271,8.667
421568,45,98,2012-10-19 00:00:00+00:00,760.01,False,B,118221,56.47,3.969,2004.02,NA,3.18,437.73,1537.49,192.330856,8.667


Krok 18: Finalne zapytanie bazowe (train + features + stores)

Połączono trzy tabele przez podwójny INNER JOIN: train z features (po Store i Date) oraz train ze stores (po Store). Wynik zawiera 421 570 wierszy — identyczną liczbę jak w oryginalnej tabeli train, co potwierdza brak utraty danych przy złączeniu (każdy wiersz sprzedaży ma odpowiadające dane ekonomiczne i informacje o sklepie). Finalna tabela zawiera 16 kolumn: identyfikatory (Store, Dept, Date), metrykę sprzedaży (Weekly_Sales), flagę święta (IsHoliday), atrybuty sklepu (store_type, store_size) oraz zmienne kontekstowe (Temperature, Fuel_Price, MarkDown1-5, CPI, Unemployment — te dwie ostatnie skonwertowane na typ liczbowy). Kolumny MarkDown1-5 pozostały jako tekst ze względu na obecność wartości 'NA' dla okresu przed listopadem 2011 — ich konwersja i obsługa braków danych zostanie wykonana w fazie Excel/Power BI.

Podsummowanie
Zaczęliśmy od eksploracji i odkrycia kilku istotnych problemów z danymi (ujemne wartości sprzedaży skoncentrowane w konkretnych działach, tekstowe "NA" w kolumnach numerycznych wymagające jawnej konwersji typów, dane wykraczające poza okres treningowy), zbadaliśmy trzy główne wymiary biznesowe (sezonowość, typ sklepu, markdowny, ekonomia), i skończyliśmy na połączonej tabeli 421 570 wierszy gotowej do dalszej analizy.

# EXCEL

Excel Krok 1: Import i czyszczenie danych źródłowych

Dane zaimportowano przez Power Query (Dane → Z tekstu/CSV → Przekształć dane), co pozwala na łatwe odświeżanie danych w przyszłości. Napotkano dwa problemy techniczne, typowe przy łączeniu źródeł z różnych ustawień regionalnych: (1) plik źródłowy zawierał nadmiarową kolumnę indeksu z eksportu CSV — usunięto ją w Power Query; (2) liczby w formacie amerykańskim (separator dziesiętny = punkt) nie były rozpoznawane przy domyślnej konwersji typu w polskich ustawieniach regionalnych Excela — rozwiązano przez opcję "Zmień typ przy użyciu ustawień regionalnych" (English-United States) dla wszystkich kolumn liczbowych. Wartości tekstowe 'NA' w kolumnach MarkDown1-5 zamieniono na puste komórki (null), a nie na 0, aby zachować rozróżnienie między "brak danych o promocji" (okres przed listopadem 2011) a "promocja o wartości zerowej" — zgodnie z wnioskiem z fazy SQL. Finalnie załadowano 421 570 wierszy, zgodnie z liczbą wierszy z zapytania bazowego SQL — brak utraty danych podczas importu.

![image-3](image-3.png)


Excel Krok 2a: Tabela przestawna — trend sprzedaży rok/miesiąc

Zbudowano tabelę przestawną grupującą Weekly_Sales po Roku i Miesiącu (pole Date, grupowanie hierarchiczne: Lata + Miesiące). Wyniki są zgodne z analizą SQL (np. luty 2010 i grudzień 2010 dają identyczne sumy w obu narzędziach), co potwierdza poprawność procesu importu danych. Wzorzec sezonowy potwierdza się w każdym z trzech lat danych: grudzień jest szczytem sprzedaży we wszystkich latach (2010: 288,8 mln; 2011: 288,1 mln), a styczeń jest najsłabszym miesiącem (2011: 163,7 mln; 2012: 168,9 mln). Sezonowość świąteczna jest więc stałym, powtarzalnym wzorcem, a nie jednorazowym zjawiskiem — co wzmacnia wcześniejszy wniosek z fazy SQL i czyni go bardziej wiarygodną podstawą do prognozowania.

![image-1](image-1.png)


Excel Krok 2b: Wykres przestawny — wizualizacja sezonowości

Stworzono wykres liniowy na bazie tabeli przestawnej, pokazujący sprzedaż tygodniową zagregowaną miesięcznie na przestrzeni trzech lat (2010-2012). Wykres uwidacznia powtarzający się wzorzec sezonowy: wyraźne piki sprzedaży w grudniu każdego roku, po których następuje równie wyraźny spadek w styczniu. Ta wizualizacja stanowi kluczowy dowód na regularność sezonowości świątecznej i będzie bazą do rekomendacji dotyczącej zwiększania poziomu zapasów przed okresem świątecznym.

![image-2](image-2.png)


Excel Krok 3a: Ranking sklepów według sprzedaży

Zbudowano tabelę przestawną sumującą Weekly_Sales dla każdego z 45 sklepów, posortowaną od największej do najmniejszej wartości. Sklep 20 generuje najwyższą sprzedaż (~301,4 mln w analizowanym okresie), a najmniejsze sklepy generują wielokrotnie mniej. Ten ranking jest bazą do analizy ABC/Pareto — sprawdzenia, jaki odsetek sklepów odpowiada za większość całkowitej sprzedaży.

![image-4](image-4.png)


Excel Krok 3b: Analiza Pareto dla sklepów (Store)

Po obliczeniu skumulowanego procentu sprzedaży w rankingu sklepów, 80% całkowitej sprzedaży generuje 27 z 45 sklepów (60%). Oznacza to, że sprzedaż jest rozłożona relatywnie równomiernie między lokalizacjami, bez silnej dominacji wąskiej grupy "topowych" sklepów. Wniosek biznesowy: strategia zarządzania zapasami i priorytetyzacji nie powinna opierać się na założeniu silnej koncentracji sprzedaży w niewielkiej grupie sklepów — większość lokalizacji wymaga porównywalnej uwagi operacyjnej.

![image-5](image-5.png)


Excel Krok 3c: Analiza Pareto dla działów (Dept)

80% sprzedaży generuje 29 z 81 działów (~36%), nie 20% jak w klasycznej zasadzie Pareto, ale wciąż silniej skoncentrowane niż w przypadku sklepów (gdzie potrzebne było 27 z 45, czyli ~60%). Top 6 działów (92, 95, 38, 72, 90, 40) odpowiada za ~33% całej sprzedaży, co wskazuje na istnienie wyraźnej grupy kluczowych kategorii produktowych. Istotne odkrycie: Dział 47 ma ujemną sumę sprzedaży w całym analizowanym okresie (-4 962,93), zgodnie z wcześniejszym wnioskiem z fazy SQL o anomalnie wysokiej liczbie zwrotów w tym dziale. Wniosek strategiczny: koncentracja sprzedaży w działach (36% działów = 80% sprzedaży) jest silniejsza niż w sklepach (51% sklepów = 80% sprzedaży), więc priorytetyzacja zarządzania zapasami powinna kładzieć większy nacisk na kategorię produktową niż na lokalizację.

**=JEŻELI([@unemployment]<7;"Low (<7%)";JEŻELI([@unemployment]<9;"Medium (7-9%)";"High (>9%)"))**

Excel Krok 4a: Utworzenie kolumny pomocniczej unemployment_tier

Dodano kolumnę kategoryzującą wartość bezrobocia na trzy przedziały: Low (<7%), Medium (7-9%), High (>9%), przy użyciu zagnieżdżonej funkcji JEŻELI w formacie tabeli strukturalnej Excela. Kategoryzacja umożliwia dalszą analizę interakcji między poziomem bezrobocia a typem sklepu w tabeli przestawnej, bez konieczności pracy na surowych wartościach liczbowych.

![image-6](image-6.png)


Excel Krok 4b: Analiza rozkładu typu sklepu według poziomu bezrobocia

Tabela przestawna z dwoma wymiarami (unemployment_tier × store_type) potwierdza hipotezę postawioną w fazie SQL: rozkład typów sklepów nie jest równomierny między przedziałami bezrobocia. Sklepy typu A (najwyższa sprzedaż) stanowią 56,9% wierszy w grupie "Medium" bezrobocia, ale tylko 35% w grupie "High" bezrobocia — gdzie dominują mniejsze sklepy typu B i C. Wniosek: wcześnio zaobserwowana wyższa średnia sprzedaż w grupie ze średnim bezrobociem jest efektem zmiennej pomieszanej (confounding variable) — różnicy w rozkładzie wielkości/typu sklepów między regionami, a nie bezpośrednim wpływem sytuacji ekonomicznej na zachowania zakupowe klientów. Surowa korelacja bezrobocie–sprzedaż była więc myląca i nie powinna być interpretowana jako związek przyczynowo-skutkowy bez uwzględnienia tego czynnika.

![image-7](image-7.png)


Excel Krok 5: Prognoza sprzedaży metodą regresji liniowej (FORECAST.LINEAR)

Zbudowano prostą prognozę sprzedaży na 3 kolejne miesiące (listopad 2012 – styczeń 2013) na bazie funkcji FORECAST.LINEAR, wykorzystującej regresję liniową na 33 punktach danych historycznych. Model przewiduje stabilny poziom sprzedaży ok. 207 mln miesięcznie, ze znikomym wzrostem trendu. Istotne ograniczenie: model regresji liniowej nie uwzględnia sezonowości — przewidywana wartość dla grudnia 2012 (miesiąc 35, ~206,8 mln) jest drastycznie niższa niż historyczny wzorzec sezonowy dla grudnia (~288 mln w 2010 i 2011 roku), ponieważ model traktuje wszystkie miesiące jednakowo, kontynuując jedynie ogólny trend długoterminowy. Wniosek: do realnego prognozowania popytu w branży o silnej sezonowości (jak handel detaliczny w okresie świątecznym) prosty model liniowy jest niewystarczający — wymagana byłaby metoda uwzględniająca sezonowość (np. dekompozycja sezonowa, wskaźniki sezonowości miesięczne, lub bardziej zaawansowane modele jak Holt-Winters), co stanowi rekomendowany kolejny krok rozwoju analizy poza zakres tego projektu.

Podsumowanie fazy Excel

W fazie Excel wykonano: (1) import i czyszczenie danych z obsługą problemów regionalnych formatu liczb i rozróżnieniem braku danych od wartości zerowych; (2) analizę trendu sezonowego potwierdzającą powtarzalność wzorca świątecznego w trzech kolejnych latach; (3) analizę ABC/Pareto pokazującą, że koncentracja sprzedaży jest znacznie silniejsza w działach (29 z 81 = 80% sprzedaży) niż w sklepach (27 z 45 = 80% sprzedaży), oraz wykrycie anomalii w Dziale 47 (ujemna suma sprzedaży); (4) weryfikację hipotezy o zmiennej pomieszanej, wykazującą że pozorny wpływ bezrobocia na sprzedaż wynika z nierównomiernego rozkładu typów sklepów między regionami; (5) prosty model prognozy liniowej, który uwidocznił ograniczenia metody nieuwzględniającej sezonowości. Wszystkie kluczowe wnioski zostały przeniesione do interaktywnego dashboardu w Power BI.

# POWER BI

Power BI Krok 1: Import danych i przygotowanie modelu

Dane zaimportowano do Power BI Desktop z pliku CSV (wyeksportowanego z DataLab) przez Power Query. Wykonano następujące przekształcenia: usunięto nadmiarową kolumnę indeksu, zamieniono wartości tekstowe 'NA' w kolumnach MarkDown1-5 na null i zmieniono ich typ na liczbę dziesiętną (z ustawieniami regionalnymi English-United States), zmieniono typ kolumny IsHoliday z logicznego na tekst i zastąpiono wartości TRUE/FALSE na Holiday/Non-Holiday. Kolumna Store została zmieniona na typ tekstowy, żeby Power BI traktował numery sklepów jako kategorie, a nie wartości liczbowe do sumowania.

Power BI Krok 2: Kolumny obliczeniowe DAX

Utworzono trzy kolumny pomocnicze w języku DAX: (1) markdown_tier — kategoryzacja MarkDown1 na przedziały High/Mid/Low/No data na podstawie wartości liczbowej; (2) markdown_tier_sort — kolumna numeryczna do sortowania markdown_tier w logicznej kolejności (zamiast alfabetycznej); (3) dept_anomalia — flaga identyfikująca Dział 47 jako anomalię ("Anomaly (Dept 47)" / "Other Departments"); (4) unemployment_tier — kategoryzacja bezrobocia na Low/Mid/High na potrzeby strony makroekonomicznej.

Power BI Krok 3: Struktura dashboardu

Zbudowano interaktywny dashboard składający się z czterech stron. Overview zawiera dwie karty KPI (łączna sprzedaż $6,74 mld, liczba sklepów 45), wykres liniowy trendu miesięcznego z automatyczną prognozą Power BI (przedział ufności 95%, sezonowość wykrywana automatycznie), wykres słupkowy Top 10 sklepów według sprzedaży oraz slicer filtrujący po roku. Seasonality & Holidays zawiera wykres sprzedaży miesięcznej z podziałem na tygodnie świąteczne i pozostałe, porównanie średniej sprzedaży dla typów sklepów A/B/C w tygodniach świątecznych vs pozostałych, oraz wykres wpływu poziomu markdownu na sprzedaż. Stores & Departments zawiera pełny ranking sklepów, osobne wykresy Top 5 i Worst 5 działów (z Działem 47 wyróżnionym kolorem czerwonym jako anomalia), wykres porównujący średnią sprzedaż typów A/B/C oraz slicer po typie sklepu. Macroeconomic Context zawiera dwie karty KPI (średnie bezrobocie 7,96%, średnie CPI 171,20), wykresy trendów CPI i bezrobocia w czasie, analizę sprzedaży według poziomu bezrobocia z podziałem na typ sklepu (wizualizacja efektu zmiennej pomieszanej) oraz trend ceny paliwa.

# Key Findings & Business Recommendations

Sezonowość świąteczna jest silna i powtarzalna. Grudzień generuje konsekwentnie ~43% więcej sprzedaży niż styczeń (najsłabszy miesiąc) we wszystkich trzech latach danych. To regularny, przewidywalny wzorzec — rekomendacja: zwiększać poziom zapasów przed grudniem, różnicując go według typu sklepu (sklepy B reagują na święta silniej procentowo niż A i C).

Koncentracja sprzedaży jest silniejsza w działach niż w lokalizacjach. 29 z 81 działów (~36%) generuje 80% całkowitej sprzedaży, podczas gdy w przypadku sklepów potrzeba 27 z 45 (~60%). Priorytetyzacja zarządzania zapasami powinna skupiać się na kategorii produktowej (szczególnie działy 92, 95, 38, 72, 90) bardziej niż na lokalizacji geograficznej.
Dział 47 wymaga pilnej uwagi operacyjnej. Jako jedyny dział wykazuje ujemną sumę sprzedaży w całym analizowanym okresie (-4 962,93), przy 254 tygodniach z ujemnymi wartościami (20% wszystkich ujemnych przypadków w zbiorze). Sugeruje to systemowy problem ze zwrotami lub błędami raportowania wymagający zbadania przez zespół merchandisingu.

Promocje (MarkDown1) korelują ze wzrostem sprzedaży, ale z malejącą efektywnością. Przejście z niskiego do średniego poziomu markdownu zwiększa średnią sprzedaż o ~36%, ale przejście ze średniego do wysokiego tylko o ~4,7%. Sugeruje to optymalny przedział inwestycji w promocje (5 000–15 000) powyżej którego zwrot maleje.

Surowa korelacja bezrobocie–sprzedaż jest myląca. Pozornie wyższa sprzedaż w regionach ze średnim bezrobociem (7-9%) wynika z nadreprezentacji dużych sklepów typu A w tych regionach (56,9% wierszy), a nie z samego poziomu bezrobocia. Decyzje operacyjne nie powinny opierać się na tej korelacji bez uwzględnienia typu/wielkości sklepu.